# Parte 2 – Modelo principal S1 e data augmentation (pessoa 2)

Nota: este notebook continua a partir do `Parte 1_testes.ipynb`. O código da pessoa 1 está copiado sem alterações em `parte2/parte1_base.py` (mesma preparação dos textos, mesmos pesos de classe, mesmas métricas e **as mesmas 5 divisões agrupadas**, seed 42). As funções novas estão em `parte2/s1.py`. Todos os resultados ficam na pasta `Parte2_resultados/` (um subdiretório por configuração, no mesmo formato da pasta `Test/` da pessoa 1).

Ordem: (1) ponto de partida, (2) classificador e C, (3) palavras vs fragmentos, (4) pesos das classes, (5) palavras-chave, (6) secções da nota, (7) pesos por campo, (8) augmentation, (9) configuração final, treino com todo o treino e modelo guardado.
Em cada passo a melhor configuração passa para o passo seguinte, escolhida com o critério `priority()` da pessoa 1 (cumprir accuracy > 0.49 e F1 mínimo ≥ 0.25 → maior macro-F1 → maior accuracy). O teste nunca é usado.

In [1]:
import sys, json
sys.path.insert(0, '../src')
import pandas as pd, numpy as np
from threadpoolctl import threadpool_limits
from parte1_base import data, splits, specialties
from s1 import *

pd.set_option('display.width', 200)
print(len(data), 'cases,', data['group_id'].nunique(), 'groups,', [len(s['evaluation']) for s in splits], 'validation cases per fold')

2613 cases, 1817 groups, [524, 529, 516, 518, 526] validation cases per fold


## 1. Ponto de partida
Melhor modelo da pessoa 1 (`all_fields_balanced_class_weights`: SVM, C=0.5, palavras em todos os campos + fragmentos 3–5 nos campos curtos com peso 0.5, pesos de classe q=1). A pessoa 1 obteve accuracy 0.591 / macro-F1 0.623 / F1 mín. 0.447; aqui é recalculado com o código do S1 para confirmar que os números coincidem.

In [2]:
best = {'classifier': 'svm', 'c': 0.5, 'q': 1, 'keywords': 'strip', 'sections': False, 'weights': default_weights('both', 0.5)}
with threadpool_limits(limits=4):
    t1 = run({'start_person1_best': best}, 'step1_start.csv')
t1

start_person1_best                         acc=0.593 macroF1=0.631 minF1=0.447 (14s)


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
start_person1_best,0.592805,0.63106,0.447257,True,5,14.487229


## 2. Classificador e parâmetro C
C é o inverso da força de regularização (C pequeno = modelo mais simples). Comparamos SVM linear e regressão logística (LR) com vários C.

In [3]:
configs = {f'svm_C{c}': {**best, 'classifier': 'svm', 'c': c} for c in [0.1, 0.5, 1]}
configs.update({f'lr_C{c}': {**best, 'classifier': 'lr', 'c': c} for c in [3, 10, 30]})
with threadpool_limits(limits=4):
    t2 = run(configs, 'step2_classifier.csv')
best = configs[best_of(t2)]
print('chosen:', best_of(t2)); t2

svm_C0.1                                   acc=0.587 macroF1=0.626 minF1=0.427 (10s)


svm_C0.5                                   acc=0.593 macroF1=0.631 minF1=0.447 (14s)


svm_C1                                     acc=0.597 macroF1=0.633 minF1=0.458 (20s)


lr_C3                                      acc=0.593 macroF1=0.616 minF1=0.426 (18s)


lr_C10                                     acc=0.591 macroF1=0.607 minF1=0.400 (21s)


lr_C30                                     acc=0.592 macroF1=0.603 minF1=0.376 (27s)


chosen: svm_C1


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
svm_C0.1,0.586682,0.625756,0.427004,True,5,9.977538
svm_C0.5,0.592805,0.63106,0.447257,True,5,14.426916
svm_C1,0.596632,0.632938,0.45815,True,5,20.145245
lr_C3,0.592805,0.616299,0.425743,True,5,17.979855
lr_C10,0.590509,0.60743,0.4,True,5,21.491368
lr_C30,0.59204,0.603129,0.375691,True,5,26.921504


## 3. Palavras vs fragmentos de palavras
`word`: só palavras (uni+bigramas); `char`: só fragmentos 3–5 caracteres; `both`: palavras + fragmentos nos campos curtos (como a pessoa 1); `both+note_char`: também fragmentos na nota clínica.

In [4]:
configs = {
    'repr_word': {**best, 'weights': default_weights('word')},
    'repr_char': {**best, 'weights': default_weights('char')},
    'repr_both': {**best, 'weights': default_weights('both', 0.5)},
    'repr_both_note_char': {**best, 'weights': default_weights('both', 0.5, transcription_char=True)},
}
with threadpool_limits(limits=4):
    t3 = run(configs, 'step3_representation.csv')
best = configs[best_of(t3)]
print('chosen:', best_of(t3)); t3

repr_word                                  acc=0.589 macroF1=0.624 minF1=0.446 (10s)


repr_char                                  acc=0.591 macroF1=0.626 minF1=0.430 (48s)


repr_both                                  acc=0.597 macroF1=0.633 minF1=0.458 (20s)


repr_both_note_char                        acc=0.596 macroF1=0.636 minF1=0.463 (75s)


chosen: repr_both_note_char


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
repr_word,0.589361,0.624418,0.446429,True,5,10.220893
repr_char,0.591274,0.625513,0.429825,True,5,48.367089
repr_both,0.596632,0.632938,0.45815,True,5,20.401483
repr_both_note_char,0.59625,0.63634,0.462882,True,5,74.93479


## 4. Pesos das classes
Peso de cada caso = (N / (12 · n_classe))^q (função `class_weights` da pessoa 1). q=0: sem pesos; q=1: balanceado; q=0.5: intermédio.

In [5]:
configs = {f'class_weights_q{q}': {**best, 'q': q} for q in [0, 0.5, 1]}
with threadpool_limits(limits=4):
    t4 = run(configs, 'step4_class_weights.csv')
best = configs[best_of(t4)]
print('chosen:', best_of(t4)); t4

class_weights_q0                           acc=0.595 macroF1=0.563 minF1=0.233 (62s)


class_weights_q0.5                         acc=0.597 macroF1=0.622 minF1=0.408 (71s)

class_weights_q1                           acc=0.596 macroF1=0.636 minF1=0.463 (75s)


chosen: class_weights_q1


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
class_weights_q0,0.595484,0.563472,0.233333,False,1,61.504067
class_weights_q0.5,0.597398,0.622417,0.408377,True,5,71.005718
class_weights_q1,0.59625,0.63634,0.462882,True,5,74.6529


## 5. Tratamento das palavras-chave
`strip`: retira a 1.ª keyword quando é o nome da especialidade (fuga da label); `none`: sem keywords; `mask`: substitui nomes de especialidades por "specialty" em todos os campos; `raw`: keywords originais, **com a fuga** — mostrado só para medir o efeito da fuga, não entra na escolha.

In [6]:
configs = {f'keywords_{k}': {**best, 'keywords': k} for k in ['strip', 'none', 'mask', 'raw']}
with threadpool_limits(limits=4):
    t5 = run(configs, 'step5_keywords.csv')
best = configs[best_of(t5.drop(index='keywords_raw'))]
print('chosen:', best_of(t5.drop(index='keywords_raw'))); t5

keywords_strip                             acc=0.596 macroF1=0.636 minF1=0.463 (75s)

keywords_none                              acc=0.595 macroF1=0.632 minF1=0.438 (60s)


keywords_mask                              acc=0.595 macroF1=0.634 minF1=0.461 (76s)


keywords_raw                               acc=0.922 macroF1=0.908 minF1=0.839 (61s)

chosen: keywords_strip


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
keywords_strip,0.59625,0.63634,0.462882,True,5,74.628432
keywords_none,0.594719,0.632231,0.4375,True,5,60.128595
keywords_mask,0.594719,0.634279,0.46087,True,5,75.71455
keywords_raw,0.922312,0.907851,0.83908,True,5,60.904349


## 6. Seleção de secções da nota
`select_sections` (pessoa 1) mantém só as secções DIAGNOSIS, IMPRESSION, ASSESSMENT, PROCEDURE, FINDINGS e PLAN da nota clínica.

In [7]:
configs = {'sections_full_note': {**best, 'sections': False}, 'sections_selected': {**best, 'sections': True}}
with threadpool_limits(limits=4):
    t6 = run(configs, 'step6_sections.csv')
best = configs[best_of(t6)]
print('chosen:', best_of(t6)); t6

sections_full_note                         acc=0.596 macroF1=0.636 minF1=0.463 (75s)


sections_selected                          acc=0.586 macroF1=0.618 minF1=0.455 (45s)

chosen: sections_full_note


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds
sections_full_note,0.59625,0.63634,0.462882,True,5,74.719903
sections_selected,0.585917,0.617815,0.455357,True,5,44.79177


## 7. Pesos por campo
Cada bloco (campo × palavras/fragmentos) é multiplicado por um peso; procura por subida coordenada em {0, 0.5, 1, 2}, avaliada nas 5 divisões com `priority()`. Peso 0 retira o bloco.

In [ ]:
with threadpool_limits(limits=4):
    fw, fw_log = field_weight_search(best)
configs = {'field_weights_before': best, 'field_weights_after': {**best, 'weights': fw}}
with threadpool_limits(limits=4):
    t7 = run(configs, 'step7_field_weights.csv')
best = configs[best_of(t7)]
print('chosen:', best_of(t7)); display(fw_log); t7

In [8]:
#Shortcut (person 3): the search in the cell above takes 25-45 min and found no improvement (see field_weight_search.csv).
#To skip it, do not run the cell above and run this one instead: it loads the configuration chosen in step 7
best = json.loads((OUT / 'field_weights_after' / 'config.json').read_text())
best

{'classifier': 'svm',
 'c': 1,
 'q': 1,
 'keywords': 'strip',
 'sections': False,
 'weights': {'description_word': 1.0,
  'description_char': 0.5,
  'sample_name_word': 1.0,
  'sample_name_char': 0.5,
  'keywords_word': 1.0,
  'keywords_char': 0.5,
  'transcription_word': 1.0,
  'transcription_char': 0.5}}

## 8. Data augmentation
Aplicada **só ao treino de cada divisão** (as variantes herdam o grupo do caso original, por isso nunca passam para a validação).
- `none`: treino original;
- `target` (classes pequenas, <100 casos no treino da divisão: Neurosurgery, Ophthalmology, Psychiatry-Psychology, Dermatology): 1 variante por *back-translation* EN→DE→EN (OPUS-MT) + 1 variante com 60% das secções da nota;
- `all` (várias classes): 1 variante com 60% das secções para cada caso de todas as classes + 30% dos casos com keywords copiados sem keywords;
- `copy` (controlo): cópias exatas de casos das classes pequenas, o mesmo número que `target` acrescenta.

In [9]:
configs = {f'augmentation_{a}': {**best, 'augmentation': a} for a in ['none', 'target', 'all', 'copy']}
with threadpool_limits(limits=4):
    t8 = run(configs, 'step8_augmentation.csv')
best = configs[best_of(t8)]
print('chosen:', best_of(t8)); t8

augmentation_none                          acc=0.596 macroF1=0.636 minF1=0.463 (175s)


augmentation_target                        acc=0.593 macroF1=0.624 minF1=0.366 (513s)


augmentation_all                           acc=0.588 macroF1=0.618 minF1=0.427 (475s)


augmentation_copy                          acc=0.595 macroF1=0.634 minF1=0.454 (276s)

chosen: augmentation_none


,accuracy,macro_f1,min_f1,meets_thresholds,folds_meeting_thresholds,seconds,augmented_cases_per_fold
augmentation_none,0.59625,0.63634,0.462882,True,5,174.703038,"[0, 0, 0, 0, 0]"
augmentation_target,0.592805,0.623899,0.366492,True,5,512.96026,"[317, 327, 310, 317, 321]"
augmentation_all,0.58783,0.618248,0.427184,True,5,475.083375,"[2368, 2358, 2368, 2373, 2351]"
augmentation_copy,0.595484,0.634275,0.453704,True,5,276.057037,"[317, 327, 310, 317, 321]"


### 8.1 Contagem e revisão das variantes
Para a divisão 0: número de variantes por tipo e por especialidade, semelhança com o texto original (Jaccard de palavras) e exemplos. Os registos completos ficam em `Parte2_resultados/augmentation_log_<modo>.csv`.

In [10]:
def jaccard(a, b):
    a, b = set(str(a).lower().split()), set(str(b).lower().split())
    return len(a & b) / max(1, len(a | b))

train0 = data[data['row_id'].isin(splits[0]['train'])]
for mode in ['target', 'all', 'copy']:
    _, log = augment(train0, mode, seed=0)
    log = log.reset_index(drop=True)  # person 3: the log repeats row labels, which recent pandas versions reject in crosstab
    original = data.set_index('row_id').loc[log['source_row_id']]
    log = log.assign(original_description=original['description'].values,
                     original_transcription=original['transcription'].values)
    log['similarity_transcription'] = [jaccard(a, b) for a, b in zip(log['transcription'], log['original_transcription'])]
    log['similarity_description'] = [jaccard(a, b) for a, b in zip(log['description'], log['original_description'])]
    log[['source_row_id', 'medical_specialty', 'augmentation', 'similarity_description', 'similarity_transcription',
         'original_description', 'description', 'keywords', 'original_transcription', 'transcription']].to_csv(OUT / f'augmentation_log_{mode}.csv', index=False)
    print(f'\n=== {mode}: {len(log)} variants (fold 0) ===')
    print(pd.crosstab(log['medical_specialty'], log['augmentation']))
    print(log.groupby('augmentation')[['similarity_description', 'similarity_transcription']].mean().round(3))


=== target: 317 variants (fold 0) ===
augmentation           backtranslation  section_crop
medical_specialty                                   
Dermatology                         20            18
Neurosurgery                        62            58
Ophthalmology                       57            38
Psychiatry-Psychology               33            31
                 similarity_description  similarity_transcription
augmentation                                                     
backtranslation                   0.631                     0.515
section_crop                      1.000                     0.467



=== all: 2368 variants (fold 0) ===
augmentation              keyword_dropout  section_crop
medical_specialty                                      
Cardiovascular-Pulmonary               54           212
Dermatology                             5            18
Gastroenterology                       40           133
General Medicine                       23           154
Neurology                              34           130
Neurosurgery                           23            58
Obstetrics-Gynecology                  25            91
Ophthalmology                          17            38
Orthopedic                             52           211
Psychiatry-Psychology                   3            31
Radiology                              58           152
Surgery                               186           620
                 similarity_description  similarity_transcription
augmentation                                                     
keyword_dropout                     1.0        

In [11]:
bt = pd.read_csv(OUT / 'augmentation_log_target.csv').query("augmentation == 'backtranslation'")
for _, r in bt.sort_values('similarity_description').head(4).iterrows():
    print(r['medical_specialty'], '| similarity', round(r['similarity_description'], 2))
    print('  original :', r['original_description'][:200])
    print('  variant  :', r['description'][:200], '\n')

Ophthalmology | similarity 0.0
  original :  Sample post dilation patient instructions.
  variant  : Patient's instructions after dilatation. 

Ophthalmology | similarity 0.15
  original :  Patient being referred for evaluation of glaucoma.
  variant  : Patients who are being used to assess glaucoma. 

Ophthalmology | similarity 0.2
  original :  Difficulty with both distance vision and with fine print at near.
  variant  : Difficulty in remote view and near fine print. 

Ophthalmology | similarity 0.2
  original :  Ophthalmology followup visit note.
  variant  : Ophthalmology follow-up. 



## 9. Configuração final, treino com todo o treino e modelo guardado

In [ ]:
import joblib
summary = pd.concat([t1, t2, t3, t4, t5, t6, t7, t8]).drop_duplicates()
summary.to_csv(OUT / 'all_results.csv')
final = dict(best)
print('Final S1 configuration:'); print(json.dumps({k: v for k, v in final.items()}, indent=1))

q = final.pop('q'); aug = final.pop('augmentation', 'none')
train_all, log_all = augment(data, aug, seed=42)
model = create_s1_model(**final)
model.fit(train_all, train_all['medical_specialty'], classifier__sample_weight=class_weights(train_all['medical_specialty'].tolist(), q))
joblib.dump(model, OUT / 's1_final_model.joblib')
(OUT / 's1_final_config.json').write_text(json.dumps({**best, 'trained_on_cases': len(data), 'augmented_cases': len(log_all)}, indent=1))
print('saved', OUT / 's1_final_model.joblib', '| trained on', len(train_all), 'rows')

In [12]:
# Sanity check for person 3: load the saved model and predict the test set (results are NOT evaluated here)
import joblib
from parte1_base import load_test
from pathlib import Path
test = load_test(ROOT / 'data' / 'test_no_labels.csv')
loaded = joblib.load(OUT / 's1_final_model.joblib')
pred = loaded.predict(test)
print(len(test), 'test cases,', len(pred), 'predictions'); print(pd.Series(pred).value_counts())

398 test cases, 398 predictions
Surgery                     116
Orthopedic                   43
Cardiovascular-Pulmonary     43
General Medicine             35
Gastroenterology             35
Radiology                    28
Neurology                    28
Neurosurgery                 21
Obstetrics-Gynecology        21
Ophthalmology                15
Psychiatry-Psychology        11
Dermatology                   2
Name: count, dtype: int64
